In [1]:
import duckdb

CSV = "../data/2025_LoL_esports_match_data_from_OraclesElixir.csv"

duckdb.sql(
    f" SELECT COUNT (*) FROM '{CSV}'"
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       120492 │
└──────────────┘

In [4]:
# import pandas as pd
# pd.set_option('display.max_rows', None)
duckdb.sql(
    f"DESCRIBE SELECT * FROM '{CSV}'"
).df()

,column_name,column_type,null,key,default,extra
0,gameid,VARCHAR,YES,None,None,None
1,datacompleteness,VARCHAR,YES,None,None,None
2,url,VARCHAR,YES,None,None,None
3,league,VARCHAR,YES,None,None,None
4,year,BIGINT,YES,None,None,None
5,split,VARCHAR,YES,None,None,None
6,playoffs,BIGINT,YES,None,None,None
7,date,TIMESTAMP,YES,None,None,None
8,game,BIGINT,YES,None,None,None
9,patch,DOUBLE,YES,None,None,None


In [9]:
duckdb.sql(f"""
    SELECT gameid, side, position, playername, champion, ban1, ban2, ban3, ban4, ban5, pick1, pick2, pick3, pick4, pick5, result
    FROM '{CSV}'
    LIMIT 24
""").df()

,gameid,side,position,playername,champion,ban1,ban2,ban3,ban4,ban5,pick1,pick2,pick3,pick4,pick5,result
0,LOLTMNT03_179647,Blue,top,PatkicaA,Gnar,Vi,Skarner,Corki,K'Sante,Sylas,NaN,NaN,NaN,NaN,NaN,0
1,LOLTMNT03_179647,Blue,jng,Joinze,Maokai,Vi,Skarner,Corki,K'Sante,Sylas,NaN,NaN,NaN,NaN,NaN,0
2,LOLTMNT03_179647,Blue,mid,Sayn,Hwei,Vi,Skarner,Corki,K'Sante,Sylas,NaN,NaN,NaN,NaN,NaN,0
3,LOLTMNT03_179647,Blue,bot,Shiganari,Jinx,Vi,Skarner,Corki,K'Sante,Sylas,NaN,NaN,NaN,NaN,NaN,0
4,LOLTMNT03_179647,Blue,sup,Lekcyc,Leona,Vi,Skarner,Corki,K'Sante,Sylas,NaN,NaN,NaN,NaN,NaN,0
5,LOLTMNT03_179647,Red,top,Nille,Renekton,Yone,Viktor,Aurora,Nocturne,Jarvan IV,NaN,NaN,NaN,NaN,NaN,1
6,LOLTMNT03_179647,Red,jng,SPOOKY,Ivern,Yone,Viktor,Aurora,Nocturne,Jarvan IV,NaN,NaN,NaN,NaN,NaN,1
7,LOLTMNT03_179647,Red,mid,Nafkelah,Orianna,Yone,Viktor,Aurora,Nocturne,Jarvan IV,NaN,NaN,NaN,NaN,NaN,1
8,LOLTMNT03_179647,Red,bot,Soldier,Varus,Yone,Viktor,Aurora,Nocturne,Jarvan IV,NaN,NaN,NaN,NaN,NaN,1
9,LOLTMNT03_179647,Red,sup,Steeelback,Braum,Yone,Viktor,Aurora,Nocturne,Jarvan IV,NaN,NaN,NaN,NaN,NaN,1


In [7]:
duckdb.sql(f"""
    SELECT patch, COUNT(DISTINCT gameid) AS n_games
    FROM '{CSV}'
    GROUP BY patch
    ORDER BY patch
""").df()

┌────────┬─────────┐
│ patch  │ n_games │
│ double │  int64  │
├────────┼─────────┤
│  15.01 │     315 │
│  15.02 │     536 │
│  15.03 │     569 │
│  15.04 │     371 │
│  15.05 │     214 │
│  15.06 │     464 │
│  15.07 │     820 │
│  15.08 │     796 │
│  15.09 │     875 │
│   15.1 │     652 │
│     ·  │      ·  │
│     ·  │      ·  │
│     ·  │      ·  │
│  15.15 │     637 │
│  15.16 │     613 │
│  15.17 │     622 │
│  15.18 │     180 │
│  15.19 │     378 │
│   15.2 │     192 │
│  15.21 │      60 │
│  15.22 │      19 │
│  15.23 │      56 │
│  15.24 │      78 │
└────────┴─────────┘
      24 rows     
     (20 shown)    

In [8]:
duckdb.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT champion, patch
        FROM '{CSV}'
        WHERE position != 'team'
    )
""").df()

,count_star()
0,3183


In [13]:
# Check if the ban's and pick's order are correct with gol.gg
df_search = duckdb.sql(f"""
    SELECT gameid, league, date, teamname, side, patch
    FROM '{CSV}'
    WHERE teamname IN ('T1', 'G2 Esports', 'Gen.G')
        AND position = 'team'
        AND league IN ('LCK', 'LEC', 'WLDs', 'MSI')
    LIMIT 2
""").df()

print(df_search)

# Replace 'YOUR_GAME_ID' with the ID you found
df_match = duckdb.sql(f"""
    SELECT side, position, teamname, champion, 
           ban1, ban2, ban3, ban4, ban5, 
           pick1, pick2, pick3, pick4, pick5
    FROM '{CSV}'
    WHERE gameid = 'LOLTMNT03_185025'
      AND position = 'team' -- Filters down to just the two team rows showing the full draft
""").df()

print(df_match)

             gameid league                date teamname  side  patch
0  LOLTMNT03_185025    LCK 2025-01-16 11:11:06       T1   Red  15.01
1  LOLTMNT03_185042    LCK 2025-01-16 11:55:28       T1  Blue  15.01
   side position   teamname champion     ban1     ban2     ban3     ban4  \
0  Blue     team  Dplus Kia     None  Kalista  Sejuani    Varus    Braum   
1   Red     team         T1     None       Vi  Skarner  K'Sante  LeBlanc   

    ban5    pick1   pick2  pick3    pick4   pick5  
0  Poppy  Nidalee  Ezreal  Leona  Ambessa   Jayce  
1    Jax     Ashe    Zyra   Yone     Rell  Gragas  
